In [1]:
# Installation des librairies nécessaires
!pip install -U openai -q

import json
import re
import time
from datetime import datetime
from pathlib import Path

from google.colab import drive, userdata
from openai import OpenAI


# ============================================================
# 1. MONTAGE DE GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive")


# ============================================================
# 2. CONNEXION À L'API OPENAI
# ============================================================

api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "La clé OPENAI_API_KEY est introuvable. "
        "Ajoute-la dans les Secrets de Google Colab."
    )

client = OpenAI(api_key=api_key)

print("Connexion à l’API configurée.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.7 MB/s eta 0:00:00
Mounted at /content/drive
Connexion à l’API configurée.


In [ ]:
# ============================================================
# 3. CONFIGURATION — ÉDITION
# ============================================================

DOSSIER = Path("/content/drive/MyDrive/Troupe 122 - 2026-27")

# Fichiers recherchés :
# Nom_OCR.txt
SUFFIXE_ENTREE = "_OCR.txt"

# Fichier final :
# Nom_EDIT.txt
SUFFIXE_SORTIE = "_EDIT.txt"

# Dossier de sauvegarde des blocs intermédiaires :
# Nom_EDIT_blocs/
SUFFIXE_DOSSIER_BLOCS = "_EDIT_blocs"

# Modèle à utiliser.
# Remplace cette valeur si ton compte API ne donne pas accès à ce modèle.
MODEL_EDITION = "gpt-5.5-2026-04-23"

# Nombre de pages OCR envoyées dans chaque appel.
# 6 à 10 pages est généralement une bonne plage.
PAGES_PAR_BLOC = 8

# Nombre maximal de nouvelles tentatives en cas d’erreur.
MAX_TENTATIVES = 4

# Pause entre deux appels.
PAUSE_ENTRE_BLOCS = 1

# Limite maximale de sortie par bloc.
# À augmenter si les pages sont très chargées.
MAX_OUTPUT_TOKENS = 16000

# Séparateur utilisé par ton notebook OCR.
SEPARATEUR_PAGE = "\n\n<<<PAGE_BREAK>>>\n\n"

# Si True, le script recommencera les blocs qui semblent contenir
# une erreur ou une sortie anormalement courte.
RETRAITER_BLOCS_SUSPECTS = True

# Rapport minimal entre la longueur de sortie et celle de l’entrée.
# Un résultat inférieur peut signaler une troncature.
RATIO_MINIMAL_LONGUEUR = 0.55

In [ ]:
# ============================================================
# 5. PROMPT D'ÉDITION OCR
# ============================================================

SYSTEM_PROMPT = """
Tu es un éditeur professionnel chargé de préparer une édition fidèle
d'une pièce de théâtre à partir d'une transcription OCR.

PRINCIPE ABSOLU

La fidélité au texte fourni est prioritaire sur toute autre considération.

Tu n'es pas un écrivain.
Tu n'es pas un correcteur stylistique.
Tu n'es pas un traducteur.

Tu ne dois jamais améliorer, réécrire, moderniser ou simplifier le texte.

CORRECTIONS AUTORISÉES

Tu peux uniquement corriger :

- les erreurs manifestes de reconnaissance OCR ;
- les caractères manifestement mal reconnus ;
- les accents manquants ;
- les apostrophes incorrectes ;
- les espaces incorrects ;
- les mots manifestement tronqués ou fusionnés ;
- les mots artificiellement coupés en fin de ligne ou de page ;
- la ponctuation manifestement détruite par l'OCR ;
- les phrases interrompues uniquement par un changement de page.

INTERDICTIONS

Tu ne dois jamais :

- reformuler une phrase ;
- corriger le style de l'auteur ;
- corriger une tournure familière ou non standard ;
- modifier le vocabulaire ;
- ajouter une information ;
- supprimer une répétition ;
- supprimer une hésitation ;
- supprimer un mot isolé ;
- modifier le rythme ;
- régulariser une syntaxe volontairement fragmentée ;
- transformer une ponctuation expressive ;
- fusionner deux paragraphes distincts ;
- supprimer un silence, une pause ou un blanc dramaturgique.

ARTEFACTS À SUPPRIMER

Supprime intégralement :

- les marqueurs [PAGE X] ;
- les marqueurs <<<PAGE_BREAK>>> ;
- les mots techniques tels que plaintext ou markdown lorsqu'ils
  proviennent du processus OCR ;
- les délimiteurs de blocs de code ;
- les numéros de pages imprimés isolés ;
- les messages automatiques d'un logiciel OCR ;
- les messages d'erreur de transcription.

Ne laisse pas de ligne vide parasite à l'endroit de leur suppression.

STRUCTURE THÉÂTRALE

Conserve rigoureusement :

- les parties ;
- les scènes ;
- les lieux ;
- les personnages ;
- les répliques ;
- les didascalies ;
- les pauses ;
- les silences ;
- les lignes isolées ;
- les répétitions ;
- les retours à la ligne dramaturgiques.

CONVENTIONS DE SORTIE

1. Titres de parties

Place les titres de parties seuls sur une ligne, entre doubles
astérisques.

Exemple :

**UN.**

2. Lieux et descriptions initiales

Place les indications de lieu ou descriptions scéniques initiales
seules sur une ligne, entre astérisques simples.

Exemple :

*Une rue. Mark et Jan.*

3. Personnages

Place le nom du personnage seul sur une ligne, en capitales, entre
doubles astérisques.

Place sa réplique immédiatement en dessous, sans astérisques.

Exemple :

**JAN.**
Mort ?

4. Didascalies

Place toutes les didascalies seules sur une ligne, entre astérisques
simples.

Exemple :

*Pause.*

*Elle sort.*

Une didascalie longue reste entièrement entre astérisques.

5. Séparateurs de scènes

Conserve les séparateurs de scène sous la forme :

***

6. Répliques

Ne transforme jamais plusieurs lignes dramaturgiques en un paragraphe
continu.

Conserve les blancs et ruptures voulus.

DOUTE OU ILLISIBILITÉ

Si un passage reste réellement impossible à lire, utilise exactement :

*[texte illisible]*

N'invente jamais une lecture.

SORTIE

Retourne uniquement le texte édité.

Ne fournis :

- aucune introduction ;
- aucune explication ;
- aucun commentaire ;
- aucun rapport ;
- aucune balise de code ;
- aucun titre ajouté par toi.
""".strip()

In [ ]:
# ============================================================
# 6. PROMPT DE RACCORD ENTRE DEUX BLOCS
# ============================================================

SYSTEM_PROMPT_RACCORD = """
Tu es un éditeur professionnel chargé de vérifier une unique jonction
entre deux blocs déjà édités d'une pièce de théâtre.

OBJECTIF UNIQUE

Tu dois examiner :

- la fin du bloc gauche ;
- le début du bloc droit.

Tu peux uniquement corriger les défauts causés par la coupure artificielle
entre les deux blocs.

CORRECTIONS AUTORISÉES

Tu peux uniquement :

- ressouder un mot coupé entre les deux blocs ;
- ressouder une phrase interrompue uniquement par la séparation des blocs ;
- supprimer un doublon créé à la jonction ;
- rétablir un espace ou une ponctuation manifestement détruits à la jonction ;
- déplacer un retour à la ligne situé artificiellement à la jonction ;
- rétablir la continuité d'une didascalie coupée entre les deux blocs.

INTERDICTIONS ABSOLUES

Tu ne dois jamais :

- réécrire une phrase ;
- reformuler ;
- corriger le style ;
- moderniser ;
- ajouter un mot absent ;
- supprimer une répétition volontaire ;
- supprimer une hésitation ;
- fusionner deux paragraphes distincts ;
- supprimer un blanc dramaturgique ;
- modifier une partie du texte éloignée de la jonction ;
- modifier la mise en forme théâtrale existante.

Si aucune correction n'est nécessaire, rends exactement les deux extraits
tels qu'ils ont été fournis.

FORMAT DE SORTIE OBLIGATOIRE

Retourne exactement :

<<<BLOC_GAUCHE>>>
texte final de l'extrait gauche
<<<FIN_BLOC_GAUCHE>>>
<<<BLOC_DROIT>>>
texte final de l'extrait droit
<<<FIN_BLOC_DROIT>>>

Ne fournis aucun commentaire, aucune explication et aucune balise de code.
""".strip()

In [ ]:
# ============================================================
# 7. LECTURE ET DÉCOUPAGE EN PAGES ET BLOCS
# ============================================================

MOTIF_PAGE = re.compile(
    r"(?=^\s*\[PAGE\s+\d+\]\s*$)",
    flags=re.MULTILINE | re.IGNORECASE,
)


def lire_fichier_ocr(chemin):
    """Lit un fichier OCR en UTF-8."""

    try:
        texte = chemin.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        texte = chemin.read_text(encoding="utf-8-sig")

    if not texte.strip():
        raise ValueError(f"Le fichier est vide : {chemin.name}")

    return texte


def decouper_en_pages(texte):
    """
    Découpe un OCR en pages.

    Ordre de priorité :
    1. séparateur <<<PAGE_BREAK>>>
    2. balises [PAGE X]
    3. texte complet considéré comme une seule unité
    """

    if "<<<PAGE_BREAK>>>" in texte:
        pages = re.split(
            r"\s*<<<PAGE_BREAK>>>\s*",
            texte,
            flags=re.IGNORECASE,
        )

    elif re.search(r"^\s*\[PAGE\s+\d+\]\s*$", texte, re.MULTILINE):
        pages = MOTIF_PAGE.split(texte)

    else:
        pages = [texte]

    pages = [
        page.strip()
        for page in pages
        if page and page.strip()
    ]

    return pages


def former_blocs(pages, pages_par_bloc):
    """
    Regroupe les pages par blocs sans modifier leur contenu.
    """

    blocs = []

    for debut in range(0, len(pages), pages_par_bloc):
        fin = min(debut + pages_par_bloc, len(pages))

        contenu = SEPARATEUR_PAGE.join(pages[debut:fin])

        blocs.append(
            {
                "numero_bloc": len(blocs) + 1,
                "page_debut": debut + 1,
                "page_fin": fin,
                "contenu": contenu,
            }
        )

    return blocs

In [ ]:
# ============================================================
# 9. APPEL API — ÉDITION D'UN BLOC
# ============================================================

def extraire_texte_reponse(response):
    """
    Extrait proprement la sortie textuelle d'une réponse API.
    """

    texte = getattr(response, "output_text", None)

    if texte and texte.strip():
        return texte.strip()

    raise RuntimeError(
        "L’API n’a renvoyé aucun texte exploitable."
    )


def editer_bloc_api(bloc, nombre_blocs):
    """
    Envoie un bloc OCR au modèle et renvoie le texte édité.
    """

    numero = bloc["numero_bloc"]
    page_debut = bloc["page_debut"]
    page_fin = bloc["page_fin"]
    texte_source = bloc["contenu"]

    message_utilisateur = f"""
Tu traites le bloc {numero} sur {nombre_blocs}.

Ce bloc provient approximativement des pages de fichier {page_debut}
à {page_fin}.

Applique strictement les instructions d'édition OCR au texte situé
entre les délimiteurs.

Le début ou la fin du bloc peut appartenir à une phrase ou à une scène
commencée dans un autre bloc. Ne complète rien qui ne figure pas dans
le texte. Ne crée aucune transition.

<DEBUT_OCR>
{texte_source}
<FIN_OCR>
""".strip()

    derniere_erreur = None

    for tentative in range(1, MAX_TENTATIVES + 1):
        try:
            response = client.responses.create(
                model=MODEL_EDITION,
                instructions=SYSTEM_PROMPT,
                input=message_utilisateur,
                max_output_tokens=MAX_OUTPUT_TOKENS,
            )

            texte = extraire_texte_reponse(response)

            return {
                "texte": texte,
                "response_id": getattr(response, "id", None),
                "tentative": tentative,
            }

        except Exception as erreur:
            derniere_erreur = erreur

            print(
                f"      Erreur tentative "
                f"{tentative}/{MAX_TENTATIVES} : {erreur}"
            )

            if tentative < MAX_TENTATIVES:
                attente = min(60, 5 * (2 ** (tentative - 1)))

                print(
                    f"      Nouvelle tentative dans "
                    f"{attente} seconde(s)..."
                )

                time.sleep(attente)

    raise RuntimeError(
        f"Échec après {MAX_TENTATIVES} tentatives : "
        f"{derniere_erreur}"
    )

In [ ]:
# ============================================================
# 11. CONTRÔLE DES SORTIES
# ============================================================

MOTIFS_INTERDITS = [
    r"<<<PAGE_BREAK>>>",
    r"^\s*\[PAGE\s+\d+\]\s*$",
    r"<DEBUT_OCR>",
    r"<FIN_OCR>",
    r"```",
    r"(?i)voici le texte",
    r"(?i)voici la version",
    r"(?i)texte corrigé",
    r"(?i)je ne peux pas",
    r"(?i)i can't assist",
]


def normaliser_pour_comptage(texte):
    """
    Retire quelques artefacts pour comparer approximativement
    les volumes d'entrée et de sortie.
    """

    texte = re.sub(
        r"<<<PAGE_BREAK>>>",
        " ",
        texte,
        flags=re.IGNORECASE,
    )

    texte = re.sub(
        r"\[PAGE\s+\d+\]",
        " ",
        texte,
        flags=re.IGNORECASE,
    )

    texte = re.sub(r"\s+", " ", texte)

    return texte.strip()


def verifier_sortie(texte_source, texte_sortie):
    """
    Retourne une liste d'avertissements.
    Une liste vide signifie qu'aucun problème évident n'a été détecté.
    """

    avertissements = []

    if not texte_sortie or not texte_sortie.strip():
        avertissements.append("sortie vide")
        return avertissements

    source_normale = normaliser_pour_comptage(texte_source)
    sortie_normale = normaliser_pour_comptage(texte_sortie)

    if source_normale:
        ratio = len(sortie_normale) / len(source_normale)

        if ratio < RATIO_MINIMAL_LONGUEUR:
            avertissements.append(
                f"sortie très courte : ratio {ratio:.2f}"
            )

    for motif in MOTIFS_INTERDITS:
        if re.search(
            motif,
            texte_sortie,
            flags=re.MULTILINE,
        ):
            avertissements.append(
                f"motif indésirable détecté : {motif}"
            )

    if texte_sortie.count("*") % 2 != 0:
        avertissements.append(
            "nombre impair d’astérisques"
        )

    return avertissements

In [ ]:
# ============================================================
# 12. NETTOYAGE TECHNIQUE MINIMAL
# ============================================================

def nettoyer_enveloppe_sortie(texte):
    """
    Retire uniquement les enveloppes techniques manifestes.
    Ne modifie pas le contenu littéraire.
    """

    texte = texte.strip()

    # Suppression éventuelle d'un bloc de code global.
    if texte.startswith("```") and texte.endswith("```"):
        lignes = texte.splitlines()

        if lignes:
            lignes = lignes[1:]

        if lignes and lignes[-1].strip() == "```":
            lignes = lignes[:-1]

        texte = "\n".join(lignes).strip()

    # Suppression des délimiteurs techniques s'ils ont été recopiés.
    texte = texte.replace("<DEBUT_OCR>", "")
    texte = texte.replace("<FIN_OCR>", "")

    return texte.strip()

In [ ]:
# ============================================================
# 13. GESTION DES BLOCS ÉDITÉS
# ============================================================

def chemin_bloc_txt(dossier_blocs, numero_bloc):
    return dossier_blocs / f"bloc_{numero_bloc:04d}.txt"


def chemin_bloc_meta(dossier_blocs, numero_bloc):
    return dossier_blocs / f"bloc_{numero_bloc:04d}.json"


def bloc_deja_valide(bloc, dossier_blocs):
    """
    Vérifie si un bloc déjà sauvegardé paraît exploitable.
    """

    chemin_txt = chemin_bloc_txt(
        dossier_blocs,
        bloc["numero_bloc"],
    )

    if not chemin_txt.exists():
        return False

    try:
        texte = chemin_txt.read_text(encoding="utf-8")
    except Exception:
        return False

    if not texte.strip():
        return False

    if not RETRAITER_BLOCS_SUSPECTS:
        return True

    avertissements = verifier_sortie(
        bloc["contenu"],
        texte,
    )

    return len(avertissements) == 0


def sauvegarder_bloc(
    dossier_blocs,
    bloc,
    texte,
    informations_api,
    avertissements,
):
    """
    Sauvegarde immédiatement le texte et ses métadonnées.
    """

    numero = bloc["numero_bloc"]

    chemin_txt = chemin_bloc_txt(
        dossier_blocs,
        numero,
    )

    chemin_meta = chemin_bloc_meta(
        dossier_blocs,
        numero,
    )

    chemin_txt.write_text(
        texte,
        encoding="utf-8",
    )

    metadonnees = {
        "numero_bloc": numero,
        "page_debut": bloc["page_debut"],
        "page_fin": bloc["page_fin"],
        "modele": MODEL_EDITION,
        "date_traitement": datetime.now().isoformat(),
        "response_id": informations_api.get("response_id"),
        "tentative_reussie": informations_api.get("tentative"),
        "longueur_entree": len(bloc["contenu"]),
        "longueur_sortie": len(texte),
        "avertissements": avertissements,
    }

    chemin_meta.write_text(
        json.dumps(
            metadonnees,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

In [ ]:
# ============================================================
# 14. FICHIERS DE RACCORD
# ============================================================

def chemin_raccord_meta(dossier_raccords, numero_raccord):
    return dossier_raccords / f"raccord_{numero_raccord:04d}.json"


def chemin_bloc_raccorde(dossier_raccords, numero_bloc):
    return dossier_raccords / f"bloc_{numero_bloc:04d}.txt"


def raccord_deja_effectue(dossier_raccords, numero_raccord):
    """
    Un raccord est considéré terminé lorsque son fichier JSON
    existe et indique le statut 'termine'.
    """

    chemin_meta = chemin_raccord_meta(
        dossier_raccords,
        numero_raccord,
    )

    if not chemin_meta.exists():
        return False

    try:
        donnees = json.loads(
            chemin_meta.read_text(encoding="utf-8")
        )

        return donnees.get("statut") == "termine"

    except Exception:
        return False

In [ ]:
# ============================================================
# 15. PASSE COMPLÈTE DE RACCORD
# ============================================================

def preparer_blocs_raccords(blocs, dossier_blocs, dossier_raccords):
    """
    Copie les blocs édités dans le dossier de raccord si leur copie
    n'existe pas encore.
    """

    dossier_raccords.mkdir(
        parents=True,
        exist_ok=True,
    )

    for bloc in blocs:
        numero = bloc["numero_bloc"]

        source = chemin_bloc_txt(
            dossier_blocs,
            numero,
        )

        destination = chemin_bloc_raccorde(
            dossier_raccords,
            numero,
        )

        if not source.exists():
            raise FileNotFoundError(
                f"Bloc édité manquant : {source.name}"
            )

        if not destination.exists():
            destination.write_text(
                source.read_text(encoding="utf-8"),
                encoding="utf-8",
            )


def effectuer_passe_raccord(
    blocs,
    dossier_blocs,
    dossier_raccords,
):
    """
    Corrige successivement toutes les jonctions entre blocs.

    Les raccords sont appliqués dans l'ordre :
    bloc 1 / bloc 2,
    puis bloc 2 / bloc 3,
    etc.

    Ainsi, le bloc droit modifié à une jonction est utilisé dans
    son état corrigé pour la jonction suivante.
    """

    preparer_blocs_raccords(
        blocs=blocs,
        dossier_blocs=dossier_blocs,
        dossier_raccords=dossier_raccords,
    )

    nombre_raccords = max(0, len(blocs) - 1)

    if nombre_raccords == 0:
        print("   Aucun raccord nécessaire : un seul bloc.")
        return

    print()
    print(f"Passe de raccord : {nombre_raccords} jonction(s)")

    for index in range(nombre_raccords):
        numero_raccord = index + 1
        numero_gauche = index + 1
        numero_droit = index + 2

        if (
            REPRENDRE_RACCORDS
            and raccord_deja_effectue(
                dossier_raccords,
                numero_raccord,
            )
        ):
            print(
                f"   Raccord {numero_raccord}/{nombre_raccords} "
                f"(blocs {numero_gauche}-{numero_droit}) : "
                f"déjà traité."
            )
            continue

        print(
            f"   Raccord {numero_raccord}/{nombre_raccords} "
            f"(blocs {numero_gauche}-{numero_droit})..."
        )

        chemin_gauche = chemin_bloc_raccorde(
            dossier_raccords,
            numero_gauche,
        )

        chemin_droit = chemin_bloc_raccorde(
            dossier_raccords,
            numero_droit,
        )

        texte_gauche = chemin_gauche.read_text(
            encoding="utf-8"
        ).strip()

        texte_droit = chemin_droit.read_text(
            encoding="utf-8"
        ).strip()

        prefixe_gauche, extrait_gauche = diviser_fenetre_fin(
            texte_gauche,
            LIGNES_CONTEXTE_RACCORD,
        )

        extrait_droit, suffixe_droit = diviser_fenetre_debut(
            texte_droit,
            LIGNES_CONTEXTE_RACCORD,
        )

        resultat = raccorder_extraits_api(
            extrait_gauche=extrait_gauche,
            extrait_droit=extrait_droit,
            numero_raccord=numero_raccord,
            nombre_raccords=nombre_raccords,
        )

        gauche_finale = resultat["gauche"]
        droite_finale = resultat["droite"]

        # Reconstruction du bloc gauche.
        parties_gauche = [
            partie
            for partie in [
                prefixe_gauche.rstrip(),
                gauche_finale.strip(),
            ]
            if partie
        ]

        nouveau_texte_gauche = "\n".join(
            parties_gauche
        ).strip() + "\n"

        # Reconstruction du bloc droit.
        parties_droites = [
            partie
            for partie in [
                droite_finale.strip(),
                suffixe_droit.lstrip(),
            ]
            if partie
        ]

        nouveau_texte_droit = "\n".join(
            parties_droites
        ).strip() + "\n"

        # Écriture immédiate pour permettre la reprise.
        chemin_gauche.write_text(
            nouveau_texte_gauche,
            encoding="utf-8",
        )

        chemin_droit.write_text(
            nouveau_texte_droit,
            encoding="utf-8",
        )

        metadonnees = {
            "statut": "termine",
            "numero_raccord": numero_raccord,
            "bloc_gauche": numero_gauche,
            "bloc_droit": numero_droit,
            "modele": MODEL_RACCORD,
            "date_traitement": datetime.now().isoformat(),
            "response_id": resultat.get("response_id"),
            "tentative_reussie": resultat.get("tentative"),
            "lignes_contexte": LIGNES_CONTEXTE_RACCORD,
            "longueur_gauche_avant": len(extrait_gauche),
            "longueur_gauche_apres": len(gauche_finale),
            "longueur_droite_avant": len(extrait_droit),
            "longueur_droite_apres": len(droite_finale),
        }

        chemin_meta = chemin_raccord_meta(
            dossier_raccords,
            numero_raccord,
        )

        chemin_meta.write_text(
            json.dumps(
                metadonnees,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

        print("      Raccord sauvegardé.")

        time.sleep(PAUSE_ENTRE_BLOCS)

In [ ]:
# ============================================================
# 16. ASSEMBLAGE DES BLOCS RACCORDÉS
# ============================================================

def assembler_blocs_raccordes(
    blocs,
    dossier_raccords,
    chemin_sortie,
):
    """
    Assemble les versions corrigées par la passe de raccord.
    """

    textes = []

    for bloc in blocs:
        numero = bloc["numero_bloc"]

        chemin_txt = chemin_bloc_raccorde(
            dossier_raccords,
            numero,
        )

        if not chemin_txt.exists():
            raise FileNotFoundError(
                f"Bloc raccordé manquant : {chemin_txt.name}"
            )

        texte = chemin_txt.read_text(
            encoding="utf-8"
        ).strip()

        textes.append(texte)

    contenu_final = "\n\n".join(
        textes
    ).strip() + "\n"

    chemin_sortie.write_text(
        contenu_final,
        encoding="utf-8",
    )

In [ ]:
# ============================================================
# 18. TRAITEMENT DU DOSSIER
# ============================================================

if not DOSSIER.exists():
    raise FileNotFoundError(
        f"Le dossier n’existe pas : {DOSSIER}"
    )

if not DOSSIER.is_dir():
    raise NotADirectoryError(
        f"Le chemin n’est pas un dossier : {DOSSIER}"
    )

fichiers_ocr = sorted(
    [
        chemin
        for chemin in DOSSIER.iterdir()
        if (
            chemin.is_file()
            and chemin.name.endswith(SUFFIXE_ENTREE)
        )
    ],
    key=lambda chemin: chemin.name.lower(),
)

print(f"Dossier scanné : {DOSSIER}")
print(f"Fichiers OCR trouvés : {len(fichiers_ocr)}")

if not fichiers_ocr:
    print(
        f"Aucun fichier se terminant par "
        f"{SUFFIXE_ENTREE} n’a été trouvé."
    )

else:
    resultats = []

    for chemin_ocr in fichiers_ocr:
        resultat = traiter_fichier_ocr(chemin_ocr)
        resultats.append(resultat)

    journal = {
        "date": datetime.now().isoformat(),
        "modele_edition": MODEL_EDITION,
        "modele_raccord": MODEL_RACCORD,
        "pages_par_bloc": PAGES_PAR_BLOC,
        "resultats": resultats,
    }

    chemin_journal = DOSSIER / "journal_edition_ocr.json"

    chemin_journal.write_text(
        json.dumps(
            journal,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print()
    print("=" * 72)
    print("RÉCAPITULATIF")

    nombre_traites = sum(
        1
        for resultat in resultats
        if resultat["statut"] == "traite"
    )

    nombre_erreurs = sum(
        1
        for resultat in resultats
        if resultat["statut"] == "erreur"
    )

    nombre_suspects = sum(
        resultat.get("suspects", 0)
        for resultat in resultats
    )

    print(f"Fichiers édités  : {nombre_traites}")
    print(f"Erreurs          : {nombre_erreurs}")
    print(f"Blocs suspects   : {nombre_suspects}")
    print(f"Journal          : {chemin_journal.name}")
    print("Terminé.")

In [ ]:
# ============================================================
# FICHIERS DE RACCORD
# ============================================================

def chemin_raccord_meta(dossier_raccords, numero_raccord):
    return dossier_raccords / f"raccord_{numero_raccord:04d}.json"


def chemin_bloc_raccorde(dossier_raccords, numero_bloc):
    return dossier_raccords / f"bloc_{numero_bloc:04d}.txt"


def raccord_deja_effectue(dossier_raccords, numero_raccord):
    """
    Un raccord est considéré terminé lorsque son fichier JSON
    existe et indique le statut 'termine'.
    """

    chemin_meta = chemin_raccord_meta(
        dossier_raccords,
        numero_raccord,
    )

    if not chemin_meta.exists():
        return False

    try:
        donnees = json.loads(
            chemin_meta.read_text(encoding="utf-8")
        )

        return donnees.get("statut") == "termine"

    except Exception:
        return False

In [ ]:
# ============================================================
# PASSE COMPLÈTE DE RACCORD
# ============================================================

def preparer_blocs_raccords(blocs, dossier_blocs, dossier_raccords):
    """
    Copie les blocs édités dans le dossier de raccord si leur copie
    n'existe pas encore.
    """

    dossier_raccords.mkdir(
        parents=True,
        exist_ok=True,
    )

    for bloc in blocs:
        numero = bloc["numero_bloc"]

        source = chemin_bloc_txt(
            dossier_blocs,
            numero,
        )

        destination = chemin_bloc_raccorde(
            dossier_raccords,
            numero,
        )

        if not source.exists():
            raise FileNotFoundError(
                f"Bloc édité manquant : {source.name}"
            )

        if not destination.exists():
            destination.write_text(
                source.read_text(encoding="utf-8"),
                encoding="utf-8",
            )


def effectuer_passe_raccord(
    blocs,
    dossier_blocs,
    dossier_raccords,
):
    """
    Corrige successivement toutes les jonctions entre blocs.

    Les raccords sont appliqués dans l'ordre :
    bloc 1 / bloc 2,
    puis bloc 2 / bloc 3,
    etc.

    Ainsi, le bloc droit modifié à une jonction est utilisé dans
    son état corrigé pour la jonction suivante.
    """

    preparer_blocs_raccords(
        blocs=blocs,
        dossier_blocs=dossier_blocs,
        dossier_raccords=dossier_raccords,
    )

    nombre_raccords = max(0, len(blocs) - 1)

    if nombre_raccords == 0:
        print("   Aucun raccord nécessaire : un seul bloc.")
        return

    print()
    print(f"Passe de raccord : {nombre_raccords} jonction(s)")

    for index in range(nombre_raccords):
        numero_raccord = index + 1
        numero_gauche = index + 1
        numero_droit = index + 2

        if (
            REPRENDRE_RACCORDS
            and raccord_deja_effectue(
                dossier_raccords,
                numero_raccord,
            )
        ):
            print(
                f"   Raccord {numero_raccord}/{nombre_raccords} "
                f"(blocs {numero_gauche}-{numero_droit}) : "
                f"déjà traité."
            )
            continue

        print(
            f"   Raccord {numero_raccord}/{nombre_raccords} "
            f"(blocs {numero_gauche}-{numero_droit})..."
        )

        chemin_gauche = chemin_bloc_raccorde(
            dossier_raccords,
            numero_gauche,
        )

        chemin_droit = chemin_bloc_raccorde(
            dossier_raccords,
            numero_droit,
        )

        texte_gauche = chemin_gauche.read_text(
            encoding="utf-8"
        ).strip()

        texte_droit = chemin_droit.read_text(
            encoding="utf-8"
        ).strip()

        prefixe_gauche, extrait_gauche = diviser_fenetre_fin(
            texte_gauche,
            LIGNES_CONTEXTE_RACCORD,
        )

        extrait_droit, suffixe_droit = diviser_fenetre_debut(
            texte_droit,
            LIGNES_CONTEXTE_RACCORD,
        )

        resultat = raccorder_extraits_api(
            extrait_gauche=extrait_gauche,
            extrait_droit=extrait_droit,
            numero_raccord=numero_raccord,
            nombre_raccords=nombre_raccords,
        )

        gauche_finale = resultat["gauche"]
        droite_finale = resultat["droite"]

        # Reconstruction du bloc gauche.
        parties_gauche = [
            partie
            for partie in [
                prefixe_gauche.rstrip(),
                gauche_finale.strip(),
            ]
            if partie
        ]

        nouveau_texte_gauche = "\n".join(
            parties_gauche
        ).strip() + "\n"

        # Reconstruction du bloc droit.
        parties_droites = [
            partie
            for partie in [
                droite_finale.strip(),
                suffixe_droit.lstrip(),
            ]
            if partie
        ]

        nouveau_texte_droit = "\n".join(
            parties_droites
        ).strip() + "\n"

        # Écriture immédiate pour permettre la reprise.
        chemin_gauche.write_text(
            nouveau_texte_gauche,
            encoding="utf-8",
        )

        chemin_droit.write_text(
            nouveau_texte_droit,
            encoding="utf-8",
        )

        metadonnees = {
            "statut": "termine",
            "numero_raccord": numero_raccord,
            "bloc_gauche": numero_gauche,
            "bloc_droit": numero_droit,
            "modele": MODEL_RACCORD,
            "date_traitement": datetime.now().isoformat(),
            "response_id": resultat.get("response_id"),
            "tentative_reussie": resultat.get("tentative"),
            "lignes_contexte": LIGNES_CONTEXTE_RACCORD,
            "longueur_gauche_avant": len(extrait_gauche),
            "longueur_gauche_apres": len(gauche_finale),
            "longueur_droite_avant": len(extrait_droit),
            "longueur_droite_apres": len(droite_finale),
        }

        chemin_meta = chemin_raccord_meta(
            dossier_raccords,
            numero_raccord,
        )

        chemin_meta.write_text(
            json.dumps(
                metadonnees,
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

        print("      Raccord sauvegardé.")

        time.sleep(PAUSE_ENTRE_BLOCS)

In [ ]:
# ============================================================
# ASSEMBLAGE DES BLOCS RACCORDÉS
# ============================================================

def assembler_blocs_raccordes(
    blocs,
    dossier_raccords,
    chemin_sortie,
):
    """
    Assemble les versions corrigées par la passe de raccord.
    """

    textes = []

    for bloc in blocs:
        numero = bloc["numero_bloc"]

        chemin_txt = chemin_bloc_raccorde(
            dossier_raccords,
            numero,
        )

        if not chemin_txt.exists():
            raise FileNotFoundError(
                f"Bloc raccordé manquant : {chemin_txt.name}"
            )

        texte = chemin_txt.read_text(
            encoding="utf-8"
        ).strip()

        textes.append(texte)

    contenu_final = "\n\n".join(
        textes
    ).strip() + "\n"

    chemin_sortie.write_text(
        contenu_final,
        encoding="utf-8",
    )

In [ ]:
# ============================================================
# 10. TRAITEMENT DU DOSSIER
# ============================================================

if not DOSSIER.exists():
    raise FileNotFoundError(
        f"Le dossier n’existe pas : {DOSSIER}"
    )

if not DOSSIER.is_dir():
    raise NotADirectoryError(
        f"Le chemin n’est pas un dossier : {DOSSIER}"
    )

fichiers_ocr = sorted(
    [
        chemin
        for chemin in DOSSIER.iterdir()
        if (
            chemin.is_file()
            and chemin.name.endswith(SUFFIXE_ENTREE)
        )
    ],
    key=lambda chemin: chemin.name.lower(),
)

print(f"Dossier scanné : {DOSSIER}")
print(f"Fichiers OCR trouvés : {len(fichiers_ocr)}")

if not fichiers_ocr:
    print(
        f"Aucun fichier se terminant par "
        f"{SUFFIXE_ENTREE} n’a été trouvé."
    )

else:
    resultats = []

    for chemin_ocr in fichiers_ocr:
        resultat = traiter_fichier_ocr(chemin_ocr)
        resultats.append(resultat)

    journal = {
        "date": datetime.now().isoformat(),
        "modele_edition": MODEL_EDITION,
        "modele_raccord": MODEL_RACCORD,
        "pages_par_bloc": PAGES_PAR_BLOC,
        "resultats": resultats,
    }

    chemin_journal = DOSSIER / "journal_edition_ocr.json"

    chemin_journal.write_text(
        json.dumps(
            journal,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print()
    print("=" * 72)
    print("RÉCAPITULATIF")

    nombre_traites = sum(
        1
        for resultat in resultats
        if resultat["statut"] == "traite"
    )

    nombre_erreurs = sum(
        1
        for resultat in resultats
        if resultat["statut"] == "erreur"
    )

    nombre_suspects = sum(
        resultat.get("suspects", 0)
        for resultat in resultats
    )

    print(f"Fichiers édités  : {nombre_traites}")
    print(f"Erreurs          : {nombre_erreurs}")
    print(f"Blocs suspects   : {nombre_suspects}")
    print(f"Journal          : {chemin_journal.name}")
    print("Terminé.")